# Modern Features & Performance

## 1. Dataclasses: Stripping the Boilerplate

In backend development, you often create classes whose sole purpose is to hold data (like a configuration object, a user profile, or an API response payload).

Writing `__init__`, `__repr__`, and `__eq__` for a class with 10 variables is tedious and error-prone. 
Modern Python (3.7+) solves this beautifully with the @dataclass decorator.
It automatically generates all the boring boilerplate code for you based on your Type Hints.

In [1]:
from dataclasses import dataclass

# Just adding this decorator tells Python to write the __init__, __repr__, 
# and __eq__ methods for us behind the scenes!
@dataclass
class APIResponse:
    # We MUST use type hints for dataclasses to work
    status_code: int
    data: dict
    message: str = "Success" # We can provide default values

# No __init__ needed!
response1 = APIResponse(200, {"user": "shubham"})
response2 = APIResponse(200, {"user": "shubham"})

# Automatically gets a beautiful __repr__
print(response1) 
# Output: APIResponse(status_code=200, data={'user': 'subham'}, message='Success')

# Automatically gets __eq__ (value comparison, not identity)
print(response1 == response2) # Output: True!

APIResponse(status_code=200, data={'user': 'shubham'}, message='Success')
True


Dataclasses are the modern standard for data-heavy structures. They keep your code incredibly clean.

## 2. Memory Optimization: `__slots__`
Remember in Module 1 when we learned that Python stores instance attributes in a hidden dictionary called `__dict__`?

Dictionaries are incredibly fast, but they consume a lot of memory. If you are building a system that creates millions of objects at runtime (e.g., parsing a massive dataset or running a high-frequency trading bot), the memory overhead of those dictionaries will crash your server.

You can tell Python not to use a dictionary, and instead allocate exactly enough fixed memory for the attributes you specify, using `__slots__`.

In [2]:
class OptimizedPoint:
    # We explicitly tell Python: "This object will ONLY ever have 'x' and 'y'. 
    # Do not create a __dict__ for it."
    __slots__ = ['x', 'y']

    def __init__(self, x, y):
        self.x = x
        self.y = y

point = OptimizedPoint(10, 20)

# This will fail! Because there is no __dict__, we cannot add new attributes dynamically.
# point.z = 30 # AttributeError: 'OptimizedPoint' object has no attribute 'z'

The tradeoff: You lose dynamic flexibility, but you save roughly 40-50% of the memory footprint per object. Use `__slots__` only when you are absolutely sure you have a performance bottleneck caused by millions of instances.

## 3. The Object Lifecycle: `__new__` vs. `__init__`
Many developers mistakenly believe `__init__` creates the object. It does not. `__init__` only initializes (sets up the state of) an object that already exists.

The method that actually allocates memory and creates the object is `__new__`.

Python calls `__new__` to carve out memory and create a blank object.

`__new__` returns that blank object.

Python immediately takes that blank object and passes it to `__init__` as the self parameter.

You rarely need to override `__new__`, but it is the secret to building certain design patterns, like the Singleton (ensuring only one instance of a class ever exists, useful for database connection pools).

In [3]:
class DatabasePoolSingleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        # When creating a new object, check if one already exists
        if cls._instance is None:
            print("Allocating memory for the first time...")
            # Use the base 'object' class to actually carve out the memory
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        # Note: __init__ will still run every time you call DatabasePoolSingleton()
        pass

pool1 = DatabasePoolSingleton()
pool2 = DatabasePoolSingleton()

print(pool1 is pool2) # Output: True (They share the exact same identity/memory address)

Allocating memory for the first time...
True
